# White matter hyperintensities: T1/FLAIR workflow

Align a local T1 to a local FLAIR and run the pretrained SYSU-Media model.
**Inputs:** a real scalar 3-D T1/FLAIR pair from the same subject. No paired
example is bundled with ANTsTorch. Set the absolute paths below or environment
variables `ANTSTORCH_EXAMPLE_T1` and `ANTSTORCH_EXAMPLE_FLAIR` before execution.
**Resources:** model weights download on first use; 3-D registration and
ensemble inference can be expensive. Registration here uses ANTsPy on CPU;
WMH inference uses ANTsTorch on the selected device.

Install ANTsTorch and notebook tools from the repository root before opening this notebook:
```bash
python -m pip install -e '.[notebooks]'
python -m ipykernel install --user --name python3 --display-name "Python 3 (ANTsTorch)"
```
Select that kernel and run the cells in order. All examples default to CPU.

In [ ]:
import torch  # Import before ANTs to avoid OpenMP conflicts on macOS.
import numpy as np
import ants
import antstorch

# Display figures inside Jupyter/VS Code even if MPLBACKEND was set to Agg.
from IPython import get_ipython
shell = get_ipython()
if shell is not None:
    shell.run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt

device = "cpu"
antstorch.set_default_device(device)

In [ ]:
import os
from pathlib import Path
t1_path = os.environ.get("ANTSTORCH_EXAMPLE_T1")
flair_path = os.environ.get("ANTSTORCH_EXAMPLE_FLAIR")
# Alternatively: t1_path = "/absolute/path/t1.nii.gz"
#              flair_path = "/absolute/path/flair.nii.gz"
for name, path in [("T1", t1_path), ("FLAIR", flair_path)]:
    if path is None or not Path(path).is_file():
        raise FileNotFoundError(f"Set an existing absolute {name} path in this cell or ANTSTORCH_EXAMPLE_{name}.")
t1 = ants.image_read(str(t1_path)).clone("float")
flair = ants.image_read(str(flair_path)).clone("float")
for image in (t1, flair):
    assert image.dimension == 3 and not image.has_components
    assert np.isfinite(image.numpy()).all() and np.ptp(image.numpy()) > 0

## Align modalities rather than copying their headers
Rigidly align the T1 to the FLAIR reference grid with ANTsPy. If the images are
already anatomically aligned, set `already_aligned=True`; resampling alone
changes the grid and does not estimate alignment. Inspect the overlay either way.

In [ ]:
already_aligned = False
output_dir = Path("example_outputs/white_matter_hyperintensities")
output_dir.mkdir(parents=True, exist_ok=True)
if already_aligned:
    aligned_t1 = ants.resample_image_to_target(t1, flair, interp_type="linear")
else:
    registration = ants.registration(
        fixed=flair, moving=t1, type_of_transform="Rigid", random_seed=42,
        outprefix=str(output_dir / "t1_to_flair_"),
    )
    aligned_t1 = registration["warpedmovout"]
assert aligned_t1.shape == flair.shape
for attribute in ("spacing", "origin", "direction"):
    np.testing.assert_allclose(getattr(aligned_t1, attribute), getattr(flair, attribute))
ants.plot(flair, overlay=aligned_t1, axis=2, nslices=6, overlay_alpha=0.4)

In [ ]:
probability = antstorch.sysu_media_wmh_segmentation(
    flair, t1=aligned_t1, use_ensemble=True, device=device, verbose=True,
)
values = probability.numpy()
assert probability.shape == flair.shape and np.isfinite(values).all()
assert values.min() >= -1e-6 and values.max() <= 1 + 1e-6
for attribute in ("spacing", "origin", "direction"):
    np.testing.assert_allclose(getattr(probability, attribute), getattr(flair, attribute))
threshold = 0.5  # Demonstration threshold; tune/validate for the intended data.
mask = ants.threshold_image(probability, threshold, 1., 1, 0)
ants.plot(flair, overlay=probability, axis=2, nslices=8,
          overlay_cmap="hot", overlay_alpha=0.6)
ants.image_write(aligned_t1, str(output_dir / "t1_in_flair.nii.gz"))
ants.image_write(probability, str(output_dir / "wmh_probability.nii.gz"))
ants.image_write(mask, str(output_dir / "wmh_mask.nii.gz"))
volume = float(np.count_nonzero(mask.numpy()) * np.prod(flair.spacing))
print(f"Thresholded volume at {threshold}: {volume:.1f} cubic physical units")
print("Outputs (matching files are overwritten):", output_dir.resolve())

The model returns a probability image, not an automatically validated lesion
burden. Inspect nonbrain signal, periventricular regions and artifacts. Brain
or white-matter masking is a separate choice; the wrapper leaves such
postprocessing to the caller. T1 can be omitted from the function call for
its FLAIR-only model, but that is a different inference configuration.